# Decision-Time & Data-Poisoning Attacks and Defenses in Machine Learning
## Benchmarking Adversarial Robustness of ML Models

**Author:** Harshit Chabria  
**Dataset:** Fashion-MNIST (10 Classes, 28x28 Grayscale Features)  
**Models Evaluated:** Logistic Regression, SVM (RBF Kernel), Random Forest, K-Nearest Neighbors, PyTorch Deep CNN  
**Clustering Anomaly Detectors:** K-Means Outlier Scoring, DBSCAN Density Filtering  
**Scope:** 20 Distinct Attacks (10 Decision-Time Evasion + 10 Data Poisoning) & 20 Mapped Defense Strategies.

In [ ]:
# 1. Install & Verify Required Dependencies
# Uncomment if running on Google Colab:
# !pip install -q torch torchvision scikit-learn seaborn matplotlib pandas scipy

import os, sys, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torchvision

print(f"[+] Python Version: {sys.version}")
print(f"[+] PyTorch Version: {torch.__version__}")
print(f"[+] CUDA Available: {torch.cuda.is_available()}")

## 1. Dataset Loading and Exploratory Data Analysis (EDA)
We load **Fashion-MNIST** featuring 10 product categories. Pixel intensities are normalized to $[0, 1]$ and flattened into 784-dimensional continuous feature vectors.

In [ ]:
from data_loader import load_data, print_eda_summary
from config import CLASS_NAMES

# Load dataset (5000 train, 1000 test for fast interactive demonstration)
data = load_data(train_samples=5000, test_samples=1000, val_samples=500)
print_eda_summary(data)

X_train, y_train = data["X_train"], data["y_train"]
X_val, y_val = data["X_val"], data["y_val"]
X_test, y_test = data["X_test"], data["y_test"]
X_train_pca, X_test_pca = data["X_train_pca"], data["X_test_pca"]
pca_model = data["pca_model"]

## 2. Model Training: 4 Classical ML Classifiers + Deep CNN Surrogate
We train:
1. **Logistic Regression (Multinomial Softmax)** - Linear decision boundary baseline
2. **Support Vector Machine (RBF Kernel)** - Non-linear maximum-margin hyperplane classifier
3. **Random Forest (100 Trees)** - Non-differentiable ensemble of axis-aligned decision trees
4. **K-Nearest Neighbors ($k=5$)** - Non-parametric instance/distance-based metric classifier
5. **Deep CNN (PyTorch)** - 2-layer convolutional network for white-box gradient computation

In [ ]:
from models import train_all_models
from evaluate import compute_classification_metrics

models = train_all_models(data, retrain_all=False)

print("\n" + "="*60)
print("        BASELINE CLEAN CLASSIFICATION METRICS")
print("="*60)
clean_baselines = {}
for m_name, model in models.items():
    metrics = compute_classification_metrics(model, X_test, y_test)
    clean_baselines[m_name] = metrics["accuracy"]
    print(f"{m_name:<22}: Accuracy = {metrics['accuracy']*100:.2f}% | Macro F1 = {metrics['f1_macro']:.4f}")

## 3. Clustering Anomaly Detectors (K-Means & DBSCAN)
We fit unsupervised clustering models on the clean training manifold to flag anomalous inference inputs:
- **K-Means Anomaly Detector**: Computes standardized Mahalanobis/Euclidean distance from assigned class cluster centroids.
- **DBSCAN Anomaly Detector**: Identifies inputs forced into low-density sparse manifolds (assigned noise label $-1$).

In [ ]:
from clustering import KMeansAnomalyDetector, DBSCANAnomalyDetector, evaluate_anomaly_detector
import attacks as atk

kmeans_det = KMeansAnomalyDetector(n_clusters=10).fit(X_train_pca)
dbscan_det = DBSCANAnomalyDetector(eps=3.0, min_samples=5).fit(X_train_pca)

# Generate test adversarial sample for clustering evaluation
X_adv_sample = atk.attack_01_fgsm_untargeted(models["Deep_CNN"], X_test[:200], y_test[:200], epsilon=0.15)
X_adv_sample_pca = pca_model.transform(X_adv_sample)

km_eval = evaluate_anomaly_detector(kmeans_det, X_test_pca[:200], X_adv_sample_pca, name="K-Means")
db_eval = evaluate_anomaly_detector(dbscan_det, X_test_pca[:200], X_adv_sample_pca, name="DBSCAN")

print(f"[*] K-Means Detector ROC-AUC: {km_eval['ROC_AUC']:.4f} | Precision: {km_eval['Precision']:.4f} | Recall: {km_eval['Recall']:.4f}")
print(f"[*] DBSCAN Detector  ROC-AUC: {db_eval['ROC_AUC']:.4f} | Precision: {db_eval['Precision']:.4f} | Recall: {db_eval['Recall']:.4f}")

## 4. Master Execution: 20 Attacks & 20 Defenses Demonstration
We execute the full benchmark covering all 20 Attack and Defense scenarios and compile the results table.

In [ ]:
# Run the complete benchmark and display master metrics table
from main import run_benchmark
run_benchmark(quick_mode=False)

from config import RESULTS_DIR
df_results = pd.read_csv(RESULTS_DIR / "summary_metrics.csv")
df_results

## 5. Visual Demonstration and Figures Gallery
Below we render the 8 publication-grade visualization artifacts generated by the benchmark.

In [ ]:
from IPython.display import Image, display
from config import FIGURES_DIR

figures = [
    ("Figure 1: Visual Gallery (Clean vs Adversarial vs Denoised)", "fig1_clean_vs_adv_gallery.png"),
    ("Figure 2: Master Performance (Baseline vs Attacked vs Defended)", "fig2_baseline_vs_attacked_vs_defended.png"),
    ("Figure 3: Attack Success Rate (ASR) Across 10 Evasion Attacks", "fig3_asr_evasion_attacks.png"),
    ("Figure 4: Poisoning Intensity Degradation Curves (0% to 30%)", "fig4_poisoning_intensity_curves.png"),
    ("Figure 5: 4 ML Classifiers Multi-Dimensional Robustness Profile", "fig5_classifier_robustness_radar.png"),
    ("Figure 6: Clustering-Assisted Anomaly Detection Scatter Plot", "fig6_clustering_anomaly_detection.png"),
    ("Figure 7: Confusion Matrices (Clean vs Attacked vs Defended)", "fig7_confusion_matrices.png"),
    ("Figure 8: Backdoor Trojan Attack & Spectral Defense", "fig8_backdoor_trojan_demo.png")
]

for title, fname in figures:
    fpath = FIGURES_DIR / fname
    if fpath.exists():
        print(f"\n{'='*70}\n{title}\n{'='*70}")
        display(Image(filename=str(fpath)))
    else:
        print(f"[!] Figure {fname} not found. Run benchmark first.")